In [ ]:
import pandas as pd
import spatialdata
import sopa
import anndata
import pathlib as pl
import scanpy as sc
import squidpy as sq
import cellcharter as cc
from lightning.pytorch import seed_everything
seed_everything(0)

import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import spatialdata as sd
import palettable

import numpy as np
import os

import scvi

from tqdm.notebook import tqdm

# Set the colors

In [ ]:
figdir = pl.Path("/add/path/here/figures/xenium/")

In [ ]:
dicts = []
colorlist = palettable.colorbrewer.sequential.Greys_9.mpl_colors
ctlist = ["T","Treg","NK"]
colormapping_lymphoid = {ct: colorlist[i+3] for i,ct in enumerate(ctlist)}
colormapping_lymphoid["B"] = colorlist[8]
dicts.append(colormapping_lymphoid)

colorlist = palettable.colorbrewer.sequential.Greens_9.mpl_colors
ctlist = ["TAM1","DC","Mast","TAM2"]
colormapping_myeloid = {ct: colorlist[i+1] for i,ct in enumerate(ctlist)}
dicts.append(colormapping_myeloid)

colorlist = palettable.colorbrewer.sequential.RdPu_9.mpl_colors
ctlist = ["Endothelial"]
colormapping_endoth = {ct: colorlist[2*i+1] for i,ct in enumerate(ctlist)}
dicts.append(colormapping_endoth)

colorlist = palettable.colorbrewer.sequential.YlOrBr_4.mpl_colors
ctlist = ["Muscle"]
colormapping_muscle = {ct: colorlist[i+1] for i,ct in enumerate(ctlist)}
dicts.append(colormapping_muscle)

colorlist = palettable.colorbrewer.sequential.Oranges_5.mpl_colors
ctlist = ["Inflammatory CAF", "Adipose CAF", "Fibroblast"]
colormapping_fibro = {ct: colorlist[i+1] for i,ct in enumerate(ctlist)}
dicts.append(colormapping_fibro)

colorlist = palettable.colorbrewer.sequential.Blues_7.mpl_colors
ctlist = ["Epithelial","Epithelial (Pre-cancerous)", "Carcinoma"]
colormapping_epi = {ct: colorlist[2*(i+1)] for i,ct in enumerate(ctlist)}
dicts.append(colormapping_epi)

colormapping = {}
for d in dicts:
    for k, v in d.items():  # d.items() in Python 3+
        colormapping.setdefault(k, []).append(v)

colormapping["Nerve/adrenal"] = [matplotlib.colors.to_rgb("pink")]
colormapping["Adipocyte"] = [matplotlib.colors.to_rgb("darkorange")]

In [ ]:
mpl_colormapping = {ct: colormapping[ct][0] for ct in colormapping}

# Get the CellCharter clusters across patients

In [ ]:
datadir = pl.Path("/add/path/here/Xenium/processed/")
all_pats = np.unique([f.stem.split("_")[1] for f in datadir.iterdir()])
all_pats = np.setdiff1d(all_pats,["adata"])
adatas = []
for pat in tqdm(all_pats):
    adata = sc.read_h5ad(datadir / f"Xenium_{pat}_annot.h5ad")
    adata.obs['patient'] = pat
    adatas.append(adata)

In [ ]:
adata = adatas[0].concatenate(*adatas[1:])

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata, 
    layer="counts", 
    batch_key='patient',
)

model = scvi.model.SCVI(adata)

In [ ]:
model.train(early_stopping=True, enable_progress_bar=True, devices=1)

In [ ]:
adata.obsm['X_scVI'] = model.get_latent_representation(adata).astype(np.float32)

In [ ]:
adata.write("/add/path/here/Xenium/processed/full_adata_annotated.h5ad")

In [ ]:
adata = sc.read_h5ad("/add/path/here/Xenium/processed/full_adata_annotated.h5ad")

In [ ]:
sq.gr.spatial_neighbors(adata, library_key='patient', coord_type='generic', delaunay=True, spatial_key='spatial', percentile=99)

In [ ]:
cc.gr.remove_long_links(adata)

In [ ]:
cc.gr.aggregate_neighbors(adata, n_layers=3, use_rep='X_scVI', out_key='X_cellcharter', sample_key='patient')

In [ ]:
model_params = {
        'random_state': 42,
        'trainer_params': {
            'accelerator':'cpu',
            'enable_progress_bar': False
        },
    }

In [ ]:
autok = cc.tl.ClusterAutoK(
    n_clusters=(3,10), 
    max_runs=5,
    convergence_tol=0.001, 
    model_class=cc.tl.GaussianMixture,
    model_params=model_params,
)

In [ ]:
autok.fit(adata, use_rep='X_cellcharter')

In [ ]:
cc.pl.autok_stability(autok)

In [ ]:
adata.obs['cluster_cellcharter'] = autok.predict(adata, use_rep='X_cellcharter')

In [ ]:
adata.write("/add/path/here/Xenium/processed/full_adata_annotated.h5ad")

# Compute enrichment of clusters

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_stacked_barplot(celltype_prop, color_palette=None, figsize=(10, 6), savepath=None):
    """
    Plots a stacked barplot showing the proportion of each cell type per cluster.

    Parameters:
    - celltype_prop: pd.DataFrame
        DataFrame where rows = cell types, columns = clusters, values = proportions (0-100).
    - color_palette: dict (optional)
        Dictionary mapping cell types to specific colors.
        Example: {"Cell Type A": "#1f77b4", "Cell Type B": "#ff7f0e"}
    - figsize: tuple
        Size of the figure (default: (10, 6)).
    - savepath: str (optional)
        Path to save the plot (default: None).

    Returns:
    - fig: matplotlib.figure.Figure
        The matplotlib figure object.
    """
    # Ensure values sum to 100% (normalize if necessary)
    celltype_prop = celltype_prop.div(celltype_prop.sum(axis=0), axis=1) * 100
    
    # Ensure last row sums exactly to 100% (fixes floating point errors)
    celltype_prop.iloc[-1] += (100 - celltype_prop.sum(axis=0))

    # Convert to long format for seaborn
    data = celltype_prop.reset_index().melt(id_vars="Cell_type", var_name="cluster_cellcharter", value_name="Proportion")

    # 🔹 Generate a color palette if none is provided
    if color_palette is None:
        unique_cell_types = data['Cell_type'].unique()
        n_colors = len(unique_cell_types)
        generated_palette = sns.color_palette("tab20", n_colors=n_colors)
        color_palette = dict(zip(unique_cell_types, generated_palette))
    
    # Create the plot
    fig, ax = plt.subplots(figsize=figsize)
    sns.set(style="whitegrid")
    
    # Initialize stacking position
    clusters = celltype_prop.columns
    n_clusters = len(clusters)
    bar_width = 0.8
    bottom = pd.Series([0] * n_clusters, index=clusters)

    for cell_type in celltype_prop.index:
        proportions = celltype_prop.loc[cell_type]
        color = color_palette.get(cell_type)

        ax.bar(
            clusters,
            proportions,
            width=bar_width,
            label=cell_type,
            color=color,
            bottom=bottom,
            edgecolor='black'  # Add edge color for better visibility
        )
        
        # Update bottom for next layer
        bottom += proportions
    
    # Customize plot appearance
    ax.set_ylabel('Proportion (%)')
    ax.set_xlabel('Cluster')
    ax.set_title('Cell Type Proportion per Cluster')

    # 🔹 Ensure Y-axis limits are exactly from 0 to 100%
    ax.set_ylim(0, 100)
    ax.set_yticks(range(0, 110, 10))
    
    # 🔹 Ensure consistent bar alignment
    ax.set_xlim(-0.5, n_clusters - 0.5)
    
    # 🔹 Remove grid for cleaner look
    ax.grid(False)
    
    # Show legend outside the plot
    ax.legend(title='Cell Type', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Save plot if requested
    if savepath:
        fig.savefig(savepath, bbox_inches="tight", dpi=300)
    
    plt.show()
    
    return fig


In [ ]:
adata = sc.read_h5ad("/add/path/here/Xenium/processed/full_adata_annotated.h5ad")

In [ ]:
cc.gr.enrichment(adata, group_key='cluster_cellcharter', label_key='Cell_type')
cc.pl.enrichment(adata, group_key='cluster_cellcharter', label_key='Cell_type', figsize=(4,4), fontsize=10, dot_scale=2)

In [ ]:
celltype_counts = adata.obs[["Cell_type","cluster_cellcharter"]].value_counts().unstack()

In [ ]:
celltype_prop = celltype_counts/celltype_counts.sum(axis=0)*100

In [ ]:
celltype_prop

In [ ]:
fig = plot_stacked_barplot(celltype_prop.fillna(0), color_palette=mpl_colormapping, figsize=(3, 2.5),
                           savepath=figdir / "cellcharter_proportions.svg")


In [ ]:
sc.pp.neighbors(adata, use_rep="X_scVI")

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=["Cell_type","cluster_cellcharter","patient"], ncols=1)

# Get cNMF signatures per cluster

In [ ]:
cnmf_sig_dir = pl.Path('/add/path/here/cNMF_malignant_genes_new_cosine/')

In [ ]:
cnmf_sigs = {}
for f in cnmf_sig_dir.iterdir():
    cnmf_sigs[f.stem] = pd.read_csv(f, index_col=0).head(100).index

In [ ]:
scaled_adata = adata.copy()

scaled_adata = scaled_adata[scaled_adata.obs.Cell_type=="Carcinoma"].copy()

scaled_adata.var_names = scaled_adata.var_names.str.upper()

sc.pp.scale(scaled_adata)

for cnmf_prog in cnmf_sigs:
    scaled_adata.obs[cnmf_prog] = scaled_adata[:,scaled_adata.var_names.intersection(cnmf_sigs[cnmf_prog])].X.mean(axis=1)

In [ ]:
adata.obs = pd.concat([adata.obs,scaled_adata.obs[list(cnmf_sigs)]],axis=1)

In [ ]:
fig = sc.pl.umap(adata, color=["cNMF_1","cNMF_2","cluster_cellcharter","cNMF_3","cNMF_4","patient","cNMF_5","Cell_type",], 
                 ncols=3, cmap="vlag", vmin=-0.5, vmax=0.5, vcenter=0, return_fig=True, wspace=0.25)

fig.set_size_inches(10.5, 6.5)

In [ ]:
fig

In [ ]:
def plot_cnmf_boxplot(adata, list_clusters, color_map=None, figsize=(10, 6)):
    """
    Plots boxplots of cNMF scores for the specified clusters.

    Parameters:
    - adata: AnnData
        Annotated data matrix containing cNMF scores in adata.obs.
    - list_clusters: list
        List of cluster labels to include in the plot.
    - color_map: dict, optional
        A dictionary for color mapping of cNMF components (e.g., {'cNMF_1': 'red', ...}). If None, a default color map will be used.
    - figsize: tuple, optional
        Size of the figure (default: (10, 6)).

    Returns:
    - fig: matplotlib.figure.Figure
        The matplotlib figure object.
    """
    # Melt the data into long format for seaborn
    data_melted = adata.obs.melt(
        value_vars=['cNMF_1', 'cNMF_2', 'cNMF_3', 'cNMF_4', 'cNMF_5'],
        id_vars=['cluster_cellcharter'],
        var_name='cNMF Component',
        value_name='Score'
    )

    # Filter the data to include only the clusters in list_clusters
    data_melted = data_melted[data_melted['cluster_cellcharter'].isin(list_clusters)]
    data_melted['cluster_cellcharter']= data_melted['cluster_cellcharter'].astype(str)

    # Set default color palette if no color_map is provided
    if color_map is None:
        color_map = sns.color_palette('viridis', n_colors=5)

    # Create the plot
    fig, ax = plt.subplots(figsize=figsize)
    sns.set(style="white")

    # Plot the boxplot
    sns.boxplot(
        data=data_melted,
        x='cluster_cellcharter',  # Grouping by cluster
        y='Score',
        hue='cNMF Component',  # Differentiating by cNMF component
        order=list_clusters,
        ax=ax,
        width=0.8,
        fliersize=1,
        palette=color_map  # Use provided or default color map
    )

    # Customize plot appearance
    ax.set_title('cNMF Scores across CellCharter clusters', fontsize=15)
    ax.set_xlabel('CC Cluster', fontsize=15)
    ax.set_ylabel('Score', fontsize=15)

    ax.hlines(0, xmin=ax.get_xlim()[0], xmax=ax.get_xlim()[1], color="gray", linestyle="--")

    # Remove top and right axis lines
    sns.despine(ax=ax, top=True, right=True)

    # Remove grid lines
    ax.grid(False)

    # Remove the legend title (optional)
    ax.legend(title='cNMF', bbox_to_anchor=(1,1,0,0), frameon=False)

    # Show the plot
    plt.show()
    
    return fig


In [ ]:
import palettable
colorlist = palettable.colorbrewer.qualitative.Set1_7.mpl_colors
colormapping_mal = {"cNMF_1": colorlist[0], "cNMF_2": colorlist[1], "cNMF_3": colorlist[3], 
                    "cNMF_4": colorlist[4], "cNMF_5": colorlist[6]}
colormapping_mal["Outlier"] = "whitesmoke"
colormapping_mal["Mixed"] = "lightgrey"

In [ ]:
fig = plot_cnmf_boxplot(adata, [0, 1, 3, 7], color_map=colormapping_mal, figsize=(3, 3.5))
fig.savefig(figdir / "cNMF_scores_per_cluster.png", dpi=300, bbox_inches="tight")

In [ ]:
adata.write_h5ad("/add/path/here/Xenium/processed/full_adata_annotated.h5ad")